In [1]:
import pickle as pk
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
)

SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

## Step 1: Load eICU data

In [2]:
feature_sets = pk.load(open('../../data/feature_names.pkl', 'rb'))
patient_agg = pd.read_parquet('../../data/clean_dataset.parquet')

print("Dataset shape:", patient_agg.shape)
patient_agg.head()

Dataset shape: (135778, 2167)


,patienthealthsystemstayid,gender,age,ethnicity,hospitalid,hospitaldischargeyear,mortality_at48h,hospital_numbedscategory,hospital_region,"admissiondx_ARDS-adult respiratory distress syndrome, non-cardiogenic pulmonary edema",admissiondx_Abdomen only trauma,admissiondx_Abdomen/extremity trauma,admissiondx_Abdomen/face trauma,admissiondx_Abdomen/multiple trauma,admissiondx_Abdomen/pelvis trauma,admissiondx_Abdomen/spinal trauma,admissiondx_Ablation or mapping of cardiac conduction pathway,"admissiondx_Abscess, neurologic","admissiondx_Abscess/infection-cranial, surgery for",admissiondx_Acid-base/electrolyte disturbance,admissiondx_Addisons disease,admissiondx_Adrenal neoplasm (including pheochromocytoma),admissiondx_Adrenalectomy,admissiondx_Alcohol withdrawal,admissiondx_Amputation (non-traumatic),admissiondx_Amyotrophic lateral sclerosis,admissiondx_Anaphylaxis,"admissiondx_Anastomosis, vascular",admissiondx_Anemia,"admissiondx_Aneurysm repair, ventricular","admissiondx_Aneurysm, abdominal aortic","admissiondx_Aneurysm, abdominal aortic; with dissection","admissiondx_Aneurysm, abdominal aortic; with rupture","admissiondx_Aneurysm, dissecting aortic","admissiondx_Aneurysm, thoracic aortic","admissiondx_Aneurysm, thoracic aortic; with dissection","admissiondx_Aneurysm, thoracic aortic; with rupture","admissiondx_Aneurysm/pseudoaneurysm, other","admissiondx_Aneurysms, repair of other (except ventricular)","admissiondx_Angina, stable (asymp or stable pattern of symptoms w/meds)","admissiondx_Angina, unstable (angina interferes w/quality of life or meds are tolerated poorly)",admissiondx_Aortic and Mitral valve replacement,admissiondx_Aortic valve replacement (isolated),"admissiondx_Apnea, sleep","admissiondx_Apnea-sleep; surgery for (i.e., UPPP - uvulopalatopharyngoplasty)",admissiondx_Appendectomy,"admissiondx_Arrest, respiratory (without cardiac arrest)","admissiondx_Arteriovenous malformation, surgery for","admissiondx_Arthritis, rheumatoid","admissiondx_Arthritis, septic",...,calcium,cd 4,chloride,cortisol,creatinine,direct bilirubin,eos,ethanol,fibrinogen,folate,free T4,glucose,glucose CSF,haptoglobin,ionized calcium,lactate,lipase,lymphs,magnesium,monos,myoglobin,pH,paCO2,paO2,phosphate,platelets x 1000,polys,potassium,prealbumin,prolactin,protein CSF,protein C,protein S,reticulocyte count,salicylate,serum ketones,serum osmolality,sodium,total bilirubin,total cholesterol,total protein,transferrin,triglycerides,troponin I,troponin T,uric acid,urinary creatinine,urinary osmolality,urinary sodium,urinary specific gravity
0,128919,Female,70.0,Caucasian,59,2015,1,<100,Midwest,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,9.000000,NaN,101.500000,NaN,2.125000,NaN,0.5,NaN,NaN,NaN,NaN,113.0,NaN,NaN,NaN,NaN,NaN,12.500000,NaN,16.500000,NaN,NaN,NaN,NaN,NaN,211.000000,70.5,4.100000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,139.000000,3.35,NaN,7.10,NaN,NaN,NaN,NaN,8.1,NaN,NaN,NaN,NaN
1,128927,Female,52.0,Caucasian,60,2015,0,<100,Midwest,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,8.100000,NaN,107.750000,NaN,0.700000,NaN,3.0,234.0,NaN,NaN,NaN,74.5,NaN,NaN,NaN,NaN,NaN,45.000000,1.90,7.000000,NaN,NaN,NaN,NaN,NaN,273.000000,45.0,3.750000,NaN,NaN,NaN,NaN,NaN,NaN,2.3,NaN,NaN,144.500000,0.40,NaN,7.40,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,128941,Male,68.0,Caucasian,73,2015,0,>= 500,Midwest,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,9.200000,NaN,103.500000,NaN,2.725000,0.1,0.5,NaN,NaN,NaN,NaN,151.5,NaN,NaN,NaN,1.666667,NaN,3.500000,1.20,4.500000,NaN,NaN,NaN,NaN,NaN,265.500000,91.5,4.300000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,134.500000,0.40,NaN,7.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.018
3,128943,Male,71.0,Caucasian,67,2015,0,None,Midwest,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,9.000000,NaN,98.000000,NaN,0.865000,NaN,0.0,NaN,NaN,NaN,NaN,153.0,NaN,NaN,NaN,0.800000,NaN,3.000000,NaN,4.500

In [10]:
# This distinction of numerical (continuous), binary, and categorical
# columns may be useful throughout the project.
all_num_cols = (
    feature_sets['labs'].tolist()
    + feature_sets['vitals'].tolist()
    + ['age', 'admissionheight', 'admissionweight']
)

all_bin_cols = (
    feature_sets['icd10_before24h'].tolist()
    + feature_sets['admissiondx'].tolist()
)

all_cat_cols = [
    # 'hospital_region',
    'ethnicity',
    'gender',
    'hospital_numbedscategory',
    'hospitaldischargeyear',
    # 'hospitalid'
]

In [7]:
# For privacy, eICU records patients older than 89 using the string "> 89".
if 'age' in patient_agg.columns:
    patient_agg.loc[patient_agg.age == '> 89', 'age'] = 90
    patient_agg['age'] = patient_agg['age'].astype(float)

## Step 1: Investigate possible prediction outcomes Y

In [8]:
# TODO: Consult week1 notebook. hint: use the features in the set 'icd10_after24h'
# Y must occur AFTER the time window used to construct X. Why?
#
# Check the Week 1 notebook before choosing this.
OUTCOME_COL = 'mortality_at48h' #CHANGE HERE
patient_agg[OUTCOME_COL].value_counts(dropna=False)

mortality_at48h
0    130565
1      5213
Name: count, dtype: int64

## Step 2: Choose X
Think: What are variables that are realistic and useful in predicting Y? For example, we probably wouldn't use a lot of hospital variables (i.e. region of hospital) to predict mortality, but we might use lab measurements.


In [11]:

# X_num = #TODO 
# X_bin = #TODO 
# X_cat = #TODO 


X_num = [c for c in all_num_cols if c in patient_agg.columns]
X_bin = [c for c in all_bin_cols if c in patient_agg.columns]
X_cat = [c for c in all_cat_cols if c in patient_agg.columns]

X_cols = X_num + X_bin + X_cat
assert 'hospitalid' not in X_cols

## Step 3: Understand different hospital statistics & Remove ineligible hospitals

In [ ]:
# TODO: Summarize sample size and outcome prevalence by hospital across all samples. 
# You can summarize other statistics too
# What does it mean if a hospital has a really really low Y prevalance, or a really small sample size? 

In [ ]:
# eg. these numbers are random but examples of how to threshold which 
# hospitals are "eligible" for our analysis
# MIN_PATIENTS_PER_HOSPITAL = 300
# MIN_Y_FREQ = .001
# remove all patients from the dataset that are in super super outier hospitals based on
# your defined eligibility criteria 

## Step 4: Explore / pick possible hospital train/heldout splits

In [ ]:
# Pick some K, maybe 10. Then try splitting by hospital region, number of beds, for K hospitals in the training data. 
# Suggested requirements:
#
# n_train_patients > 10,000
# 5 <= n_heldout_hospitals < 15
#
# PLUS enough positive and negative outcomes in every held-out hospital.

In [ ]:
## Create train / heldout data split
#  DOUBLE check there is no overlap of heldout data with train data!!

In [ ]:
train_df = .... #all patients in the training hospitals
#note: # For the $K$ training hospitals selected, you'll need someway to tune the models so consider making an internal validation dataset
# using the training dataset. Or you can use cross validation.
heldout_df = ... 

## Step 5: Build a simple baseline model 

In [ ]:
# First, preprocess the data, ie. imputation, one hot encoding, scaling. Use sklearn.

In [ ]:
# Define the model, XGBoost or Logistic Regression. Use sklearn.

In [ ]:
# Learn on the training data. Eg. 
# baseline_model.fit(
#     train_df[X_cols],
#     train_df[OUTCOME_COL]
# )